In [32]:
"""
Perceived access difficulties by parental status & gender
Perceive differences in policy support for families by the government by parental status & gender
- Survey-weighted means and percentages (w4weight)
- Scale: 1 = Extremely difficult, 4 = Easy
- Non-parents restricted to age 25–45
- NA codes excluded: 6, 7, 8, 9, 66, 77, 88, 99, 6666, 7777, 8888, 9999
"""

'\nPerceived access difficulties by parental status & gender\nPerceive differences in policy support for families by the government by parental status & gender\n- Survey-weighted means and percentages (w4weight)\n- Scale: 1 = Extremely difficult, 4 = Easy\n- Non-parents restricted to age 25–45\n- NA codes excluded: 6, 7, 8, 9, 66, 77, 88, 99, 6666, 7777, 8888, 9999\n'

In [33]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import matplotlib.patheffects as PathEffects 

In [34]:
# 1. DATA LOADING
df = pd.read_csv("/Users/bonnie/i4ng-datathon/r-project/data/ess1011_cronos3_withchild.csv")
# Clean & filter

VARS = ["gov_should_spend_family"]
NA_CODES = {6, 7, 8, 9, 66, 77, 88, 99, 6666, 7777, 8888, 9999}

# Define weight based on your latest file
df = df[df["w4weight"].notna()]
df = df[df["w4weight"] > 0]
df["weight"] = df["w4weight"]

/var/folders/3s/_fmfjj6s2jb_g8kwm2gtv6v40000gn/T/ipykernel_47298/367260702.py:2: DtypeWarning: Columns (18,20,87,89,813,815,819,820,821,825,826,1163,1164,1165,1166,1167,1168,1169,1170,1171,1173,1178) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/bonnie/i4ng-datathon/r-project/data/ess1011_cronos3_withchild.csv")


In [35]:
# 2. Replace NA codes with NaN in all access variables
for v in VARS:
    df[v] = df[v].where(~df[v].isin(NA_CODES), other=np.nan)

In [36]:
# 4. Group definition
#    Parents: has_children_u18 == True (all ages)
#    Non-parents: has_children_u18 == False, age 25–40
df = df[df["gndr.x"].isin([1, 2])]
df["gndr_label"] = df["gndr.x"].map({1: "Men", 2: "Women"})

In [37]:
parent_mask     = df["has_children_u18"] == True
nonparent_mask  = (df["has_children_u18"] == False) & df["agea"].between(25, 40)
df_plot         = df[parent_mask | nonparent_mask].copy()

In [38]:
df_plot["group"] = np.where(
    df_plot["has_children_u18"],
    df_plot["gndr_label"] + "\n(parent)",
    df_plot["gndr_label"] + "\n(non-parent, 25–40)"
)

In [39]:
GROUP_ORDER = [
    "Men\n(parent)",
    "Women\n(parent)",
    "Men\n(non-parent, 25–40)",
    "Women\n(non-parent, 25–40)",
]

In [40]:
print("Group sizes (unweighted):")
print(df_plot["group"].value_counts())
print("\nNaN rates per variable:")
print(df_plot[VARS].isna().mean().round(3))

Group sizes (unweighted):
group
Women\n(parent)               1443
Men\n(parent)                 1033
Women\n(non-parent, 25–40)     503
Men\n(non-parent, 25–40)       457
Name: count, dtype: int64

NaN rates per variable:
gov_should_spend_family    0.143
dtype: float64


In [41]:
# ── WEIGHTED AGGREGATION HELPERS ──────────────────────────────────────────────
def weighted_mean(series, weights):
    """Weighted mean, ignoring NaN in series."""
    mask = series.notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(series[mask], weights=weights[mask])

In [42]:
def weighted_se(series, weights):
    """Approximate weighted SE (Taylor linearisation for simple cases)."""
    mask = series.notna()
    if mask.sum() < 2:
        return np.nan
    wm  = weighted_mean(series, weights)
    w   = weights[mask].values
    v   = series[mask].values
    # Effective-sample SE approximation
    n_eff = (w.sum() ** 2) / (w ** 2).sum()
    var_w = np.average((v - wm) ** 2, weights=w)
    return np.sqrt(var_w / n_eff)

In [43]:
def weighted_pct(series, weights, cat):
    """Weighted percentage for category cat, ignoring NaN."""
    mask = series.notna()
    if mask.sum() == 0:
        return np.nan
    w_total = weights[mask].sum()
    w_cat   = weights[mask & (series == cat)].sum()
    return 100 * w_cat / w_total

In [44]:
# ── COMPUTE SUMMARIES ─────────────────────────────────────────────────────────
var_labels  = ["Goveernment should\n spend more on families"]
cat_labels  = ["0 – Disagree", "1 – Agree"]

In [45]:
# Means + CI
mean_records = []
for grp in GROUP_ORDER:
    sub = df_plot[df_plot["group"] == grp]
    for vi, var in enumerate(VARS):
        m  = weighted_mean(sub[var], sub["weight"])
        se = weighted_se(sub[var],   sub["weight"])
        mean_records.append({
            "group": grp, "var_label": var_labels[vi],
            "mean": m, "lo": m - 1.96*se, "hi": m + 1.96*se
        })
mean_df = pd.DataFrame(mean_records)

In [46]:
# Percentage distributions
pct_records = []
for grp in GROUP_ORDER:
    sub = df_plot[df_plot["group"] == grp]
    for vi, var in enumerate(VARS):
        for cat in [0, 1]:
            p = weighted_pct(sub[var], sub["weight"], cat)
            pct_records.append({
                "group": grp, "var": var, "var_label": var_labels[vi],
                "cat": cat, "pct": p
            })
pct_df = pd.DataFrame(pct_records)

In [ ]:
# ─── DESIGN SYSTEM ─────────────────────────────────────────────────────────────
PALETTE = {
    "Men\n(parent)":                "#1B4F8A",
    "Women\n(parent)":              "#5B9BD5",
    "Men\n(non-parent, 25–40)":     "#C0392B",
    "Women\n(non-parent, 25–40)":   "#E8836D",
}
LINE_STYLES = {
    "Men\n(parent)":                "solid",
    "Women\n(parent)":              "dashed",
    "Men\n(non-parent, 25–40)":     "solid",
    "Women\n(non-parent, 25–40)":   "dashed",
}
CAT_COLORS  = ["#2C3E6B", "#5B7FBD", "#FFB98387", "#C0392B"]
LIGHT_GRAY  = "#F5F5F2"
MID_GRAY    = "#C8A832"
DARK_TEXT   = "#1A1A2E"
ANNOT_GRAY  = "#6B6B6B"

In [48]:
plt.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Georgia", "Times New Roman", "DejaVu Serif"],
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.spines.left":  False,
    "axes.grid":         True,
    "axes.grid.axis":    "x",
    "grid.color":        MID_GRAY,
    "grid.linewidth":    0.6,
    "text.color":        DARK_TEXT,
    "axes.labelcolor":   DARK_TEXT,
    "xtick.color":       ANNOT_GRAY,
    "ytick.color":       DARK_TEXT,
})

In [49]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Weighted mean comparison
# ══════════════════════════════════════════════════════════════════════════════
fig1, ax = plt.subplots(figsize=(9, 5.5))
fig1.patch.set_facecolor(LIGHT_GRAY)
ax.set_facecolor(LIGHT_GRAY)

In [50]:
x       = np.arange(len(var_labels))
offsets = np.linspace(-0.28, 0.28, len(GROUP_ORDER))

In [51]:
for i, grp in enumerate(GROUP_ORDER):
    sub = mean_df[mean_df["group"] == grp].set_index("var_label")
    m   = [sub.loc[v, "mean"] for v in var_labels]
    lo  = [sub.loc[v, "lo"]   for v in var_labels]
    hi  = [sub.loc[v, "hi"]   for v in var_labels]

    ax.plot(x + offsets[i], m,
            color=PALETTE[grp], linewidth=1.4,
            linestyle=LINE_STYLES[grp],
            marker="o", markersize=7,
            markerfacecolor=PALETTE[grp],
            markeredgecolor="white", markeredgewidth=1.2,
            zorder=4, label=grp.replace("\n", " "))
    ax.errorbar(x + offsets[i], m,
                yerr=[np.array(m) - np.array(lo),
                      np.array(hi) - np.array(m)],
                fmt="none", ecolor=PALETTE[grp],
                elinewidth=1.2, capsize=4, capthick=1.2,
                zorder=3, alpha=0.7)
    for j, val in enumerate(m):
        if i == 0:
            # First group (Left dot): Shift text slightly left, align right
            txt = ax.text(x[j] + offsets[i], val+0.1, f"{val:.2f}", 
                           ha="right", va="center", color=PALETTE[grp], 
                           fontsize=9, fontweight="bold", zorder=5)
        else:
            # Second group (Right dot): Shift text slightly right, align left
            txt = ax.text(x[j] + offsets[i], val-0.2, f"{val:.2f}", 
                           ha="left", va="center", color=PALETTE[grp], 
                           fontsize=9, fontweight="bold", zorder=5)
            
        # # Add a white "halo" outline to the text so lines passing behind don't obscure it
        txt.set_path_effects([PathEffects.withStroke(linewidth=0.9, foreground=LIGHT_GRAY)])

In [52]:
ax.axhline(2.5, color=MID_GRAY, linewidth=1.2, linestyle=":", zorder=1)
ax.text(len(var_labels) - 0.05, 2.54, "midpoint",
        fontsize=7.5, color=ANNOT_GRAY, ha="right", style="italic")

Text(0.95, 2.54, 'midpoint')

In [53]:
ax.set_xticks(x)
ax.set_xticklabels(var_labels, fontsize=11)
ax.set_yticks([0, 1])
ax.set_yticklabels(["0 - Disagree", "1 - Agree"],
                   fontsize=9, color=ANNOT_GRAY)
# ax.set_ylim(0.7, 4.5)
ax.set_ylim(0.3, 4.0)
ax.set_xlim(-0.6, len(var_labels) - 0.4)

ax.spines["bottom"].set_color(MID_GRAY)
ax.tick_params(axis="x", length=0, pad=8)

In [54]:
ax.legend(frameon=True, framealpha=0.9,
          facecolor="white", edgecolor=MID_GRAY,
          fontsize=9, loc="upper right",
          title="Group  (solid = men, dashed = women)",
          title_fontsize=8.5)

In [55]:
fig1.suptitle("Government should spend more on families?",
              fontsize=15, fontweight="bold", x=0.05, ha="left",
              color=DARK_TEXT, y=0.97)
fig1.text(0.05, 0.90, 
          "Survey-weighted means · 95% CI · 9/missing excluded · non-parents restricted to ages 25–40",
          fontsize=8.5, color=ANNOT_GRAY, ha="left")
fig1.text(0.98, 0.01, "Source: ESS10 / ESS11 / CRONOS 3  |  Weighted by w4weight",
          ha="right", fontsize=7.5, color=ANNOT_GRAY, style="italic")

Text(0.98, 0.01, 'Source: ESS10 / ESS11 / CRONOS 3  |  Weighted by w4weight')

In [56]:
fig1.tight_layout(rect=[0, 0.03, 1, 0.95])
fig1.savefig("/Users/bonnie/i4ng-datathon/r-project/data/fig3_final.png",
             dpi=220, bbox_inches="tight", facecolor=LIGHT_GRAY)
print("Fig 3 saved.")

Fig 3 saved.


In [916]:
# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Weighted stacked percentage bars
# ══════════════════════════════════════════════════════════════════════════════
fig2, axes = plt.subplots(1, len(ACCESS_VARS), figsize=(13, 5.5), sharey=True)
fig2.patch.set_facecolor(LIGHT_GRAY)

short_labels = [g.replace("\n", "\n") for g in GROUP_ORDER]

In [926]:
for col_idx, (var, var_label) in enumerate(zip(ACCESS_VARS, var_labels)):
    ax2 = axes[col_idx]
    ax2.set_facecolor(LIGHT_GRAY)
    for spine in ["top", "right", "left"]:
        ax2.spines[spine].set_visible(False)
    ax2.spines["bottom"].set_color(MID_GRAY)

    y     = np.arange(len(GROUP_ORDER))
    lefts = np.zeros(len(GROUP_ORDER))

    sub = pct_df[pct_df["var"] == var]
    for cat_idx, cat in enumerate([1, 2, 3, 4]):
        # Match current loop logic to the 4 specific group names in short_labels
        vals = []
        for grp_name in short_labels:
            match = sub[(sub["group"] == grp_name) & (sub["cat"] == cat)]["pct"]
            vals.append(match.values[0] if not match.empty else 0)
        
        vals = np.array(vals)
        ax2.barh(y, vals, left=lefts, height=0.55,
                 color=CAT_COLORS[cat_idx], zorder=2)
        
        for j, (v, l) in enumerate(zip(vals, lefts)):
            if v > 7: 
                ax2.text(l + v/2, j, f"{v:.0f}%", 
                         ha="center", va="center", 
                         color="white", fontweight="bold", fontsize=9)
        lefts += vals

        # for j, (v, l) in enumerate(zip(vals, lefts)):
        #     if v >= 9:
        #         ax2.text(l + v / 2, j, f"{v:.0f}%",
        #                  ha="center", va="center",
        #                  fontsize=8.5, color="white", fontweight="bold")
        # lefts += vals

    ax2.set_xlim(0, 100)
    ax2.set_xticks([0, 25, 50, 75, 100])
    ax2.xaxis.set_major_formatter(mticker.PercentFormatter())
    ax2.tick_params(axis="x", labelsize=8, colors=ANNOT_GRAY, length=3)
    ax2.set_title(var_label, fontsize=11, fontweight="bold",
                  color=DARK_TEXT, pad=10, loc="left")
    ax2.grid(axis="x", color=MID_GRAY, linewidth=0.5, zorder=0)

    if col_idx == 0:
        ax2.set_yticks(y)
        ax2.set_yticklabels(short_labels, fontsize=8, color=DARK_TEXT, fontweight="bold")
        # ax2.tick_params(axis="y", length=0, pad=12, labelleft=True) # Forces label visibility
    else:
        ax2.tick_params(axis="y", length=0, labelleft=False)

    # if col_idx == 0:
    #     ax2.set_yticks(y)
    #     ax2.set_yticklabels(short_labels, fontsize=10, fontweight="bold", color=DARK_TEXT)
    #     # pad=10 pushes the text slightly left so it doesn't touch the bars
    #     ax2.tick_params(axis="y", length=0, pad=10) 
    # else:
    #     # Keep the other charts completely empty on the left
    #     ax2.set_yticks(y)
    #     ax2.set_yticklabels([])
    #     ax2.tick_params(axis="y", length=0)

In [927]:
legend_patches = [mpatches.Patch(color=CAT_COLORS[i], label=cat_labels[i])
                  for i in range(4)]
fig2.legend(handles=legend_patches, loc="lower center", ncol=4,
            fontsize=9, frameon=True, facecolor="white", edgecolor=MID_GRAY,
            bbox_to_anchor=(0.5, -0.05))


In [928]:
fig2.suptitle("Response distribution: How difficult is it to access family services?",
              fontsize=14, fontweight="bold", x=0.02, ha="left",
              color=DARK_TEXT, y=1.01)
fig2.text(0.02, 0.94,
          "Survey-weighted % · 9/missing excluded · non-parents restricted to ages 25–40",
          fontsize=9, color=ANNOT_GRAY, ha="left")
fig2.text(0.99, -0.07, "Source: ESS10 / ESS11 / CRONOS 3  |  Weighted by w4weight",
          ha="right", fontsize=7.5, color=ANNOT_GRAY, style="italic")

Text(0.99, -0.07, 'Source: ESS10 / ESS11 / CRONOS 3  |  Weighted by w4weight')

In [929]:
fig2.tight_layout(rect=[0.02, 0.07, 1, 0.95])
fig2.savefig("/Users/bonnie/i4ng-datathon/r-project/data/fig2_final.png",
             dpi=220, bbox_inches="tight", facecolor=LIGHT_GRAY)
print("Fig 2 saved.")

Fig 2 saved.
